# Model Training Framework 3
#### Code for training three model architectures. Compared to Model Training Framework 2 batch normalization is applied after every convolution layer.

In [1]:
# Import necessary modules
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import os
from os import listdir
from os.path import isfile, join
from PIL import Image
import keras
from keras import layers
import csv
from tensorflow.keras.utils import to_categorical

2025-10-31 18:07:53.938549: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761934074.150310      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761934074.208495      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
# Read the data set 
eda = pd.read_csv('train_labels.csv')
eda

,id,label
0,f38a6374c348f90b587e046aac6079959adf3835,0
1,c18f2d887b7ae4f6742ee445113fa1aef383ed77,1
2,755db6279dae599ebb4d39a9123cce439965282d,0
3,bc3f0c64fb968ff4a8bd33af6971ecae77c75e08,0
4,068aba587a4950175d04c680d38943fd488d6a9d,0
...,...,...
220020,53e9aa9d46e720bf3c6a7528d1fca3ba6e2e49f6,0
220021,d4b854fe38b07fe2831ad73892b3cec877689576,1
220022,3d046cead1a2a5cbe00b2b4847cfb7ba7cf5fe75,0
220023,f129691c13433f66e1e0671ff1fe80944816f5a2,0


In [3]:
# Function to select a train set sample, because the 0/1 division is 60/40
# the sample is stratified this way
def sub_sample(frac_):
    zero_sample = int(len(eda)*frac_*0.6)
    eda_zero = eda[eda.label == 0]
    sample_eda_zero = eda_zero.sample(n = zero_sample, replace=False, random_state=52)

    # A sub sample of only one labels  
    one_sample = int(len(eda)*frac_*0.4)
    eda_one = eda[eda.label == 1]
    sample_eda_one = eda_one.sample(n = one_sample, replace=False, random_state=52)

    # Merge the sample data sets 
    frames = [sample_eda_zero, sample_eda_one]
    train_df = pd.concat(frames)

    # Shuffle the observations in random order 
    train_df = train_df.sample(frac=1, random_state = 126)
    
    return train_df

In [4]:
# Select 10% of observations
train_df = sub_sample(0.1)
train_df

,id,label
176288,1d122a55476973f3847a705c96817c1e1b5ede9d,0
155346,5d94c8b0f0645c894c0ffcb292bbceb70393ea20,0
48863,d505c8b4fc7d65596ff3a8e6197b3a2474b68371,0
175558,11f7a527b4cabfcc4d00b455c9ecb52fc0547feb,1
195137,36ddfecad2e2b330e7bc8ba8fde52ddd2300168e,0
...,...,...
177151,db01a27193d200bc6d626f22d40fa2a441ebe082,1
211211,b3ed317e4997ad9435d2a8f9aeddcdb16fff079f,0
28330,f2552d8e74f0c4ccead7af168b9c3d5e2ce94bce,0
5173,112e7f5aff9bb19cfe15e8646ba6a5bcf7dbe4fd,0


In [5]:
def to_array(train_df = train_df):

    # String of working directory
    train_dir = '/kaggle/input/histopathologic-cancer-detection/train'
    
    imarray_totaal = []
    # Loop to import training images n train_df (by file names) from working directory
    for j in train_df.id:
    
        im = Image.open(train_dir + '/' + j + '.tif')
        # Turn '.tif' file into array
        imarray = np.array(im)
        
        imarray_totaal.append(imarray/255) # Normalize RGB values
    
    # Turn list into array
    imarray_totaal= np.array(imarray_totaal)
    
    # One hot encoding for the label: Turning one label (0/1) into two labels (Label_0 and Label_1) 
    num_classes = 2
    labels = to_categorical(train_df.label, num_classes=num_classes)
    
    # Giving imarray_totaal (x_train) and labels (y_train) familiar names 
    x_train = imarray_totaal
    y_train = labels

    return x_train, y_train

In [6]:
# Function to write model training results to an external location
def write_away(name, model_results, path):

# Writing away the results 
    name = pd.DataFrame.from_dict(model_results)
    name.to_csv(path, index=False)
    return name

In [7]:
# Apply to_array
train_set = to_array()
x_train = train_set[0]
y_train = train_set[1]

### Model 1

In [8]:
num_classes = 2
input_shape = (96, 96, 3)

model = keras.Sequential(
    [keras.Input(shape = input_shape),
     layers.Conv2D(filters = 32, kernel_size=(3,3),activation = 'sigmoid'),
     layers.BatchNormalization(),
     layers.MaxPooling2D(pool_size=(2, 2)),
     layers.Flatten(),
     layers.Dense(num_classes, activation="softmax"),
    ]
    )
model.summary()

I0000 00:00:1761934477.194622      37 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1761934477.195313      37 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 94, 94, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 94, 94, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 47, 47, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 70688)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │       141,378 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 142,402 (556.26 KB)

 Trainable params: 142,338 (556.01 KB)

 Non-trainable params: 64 (256.00 B)

In [9]:
# training model 1
batch_size = 200
epochs = 20
model.compile(loss="categorical_crossentropy", optimizer="sgd", metrics=["accuracy", "auc"])

history = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

Epoch 1/20


I0000 00:00:1761934511.666071     103 service.cc:148] XLA service 0x7c3db8008360 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1761934511.666932     103 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1761934511.666952     103 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1761934511.832644     103 cuda_dnn.cc:529] Loaded cuDNN version 90300


  5/100 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.5303 - auc: 0.5186 - loss: 7.3346

I0000 00:00:1761934514.386604     103 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


100/100 ━━━━━━━━━━━━━━━━━━━━ 10s 63ms/step - accuracy: 0.6050 - auc: 0.6062 - loss: 4.0574 - val_accuracy: 0.3975 - val_auc: 0.4388 - val_loss: 1.9396
Epoch 2/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.6974 - auc: 0.7529 - loss: 0.6393 - val_accuracy: 0.6025 - val_auc: 0.6998 - val_loss: 1.6325
Epoch 3/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.7294 - auc: 0.7910 - loss: 0.5806 - val_accuracy: 0.6461 - val_auc: 0.6956 - val_loss: 0.6559
Epoch 4/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.7294 - auc: 0.8005 - loss: 0.5613 - val_accuracy: 0.6034 - val_auc: 0.6701 - val_loss: 0.6497
Epoch 5/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.7590 - auc: 0.8325 - loss: 0.5103 - val_accuracy: 0.6025 - val_auc: 0.6534 - val_loss: 0.7920
Epoch 6/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.7528 - auc: 0.8224 - loss: 0.5321 - val_accuracy: 0.6025 - val_auc: 0.6696 - val_loss: 1.3582
Epoch 7/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 3

In [10]:
# Model 1: Writing away the results 
write_away('Simple_CNN_St_no', history.history, 'Simple_CNN_St_no.csv')

,accuracy,auc,loss,val_accuracy,val_auc,val_loss
0,0.656735,0.666054,1.930321,0.397547,0.438832,1.939637
1,0.721176,0.783638,0.577338,0.602453,0.699847,1.632538
2,0.740114,0.806973,0.547240,0.646070,0.695641,0.655858
3,0.747892,0.819143,0.530316,0.603362,0.670057,0.649663
4,0.768850,0.842191,0.498789,0.602453,0.653388,0.792032
5,0.773193,0.847163,0.492368,0.602453,0.669603,1.358185
6,0.792940,0.867828,0.462540,0.629714,0.724083,1.164683
7,0.796222,0.871830,0.455462,0.616084,0.658877,0.690531
8,0.814100,0.891361,0.426923,0.428896,0.393358,2.830628
9,0.800263,0.873161,0.467471,0.712858,0.780903,0.633981


### Model 2

In [11]:
num_classes = 2
input_shape = (96, 96, 3)

model1 = keras.Sequential(
    [keras.Input(shape = input_shape),
     layers.Conv2D(filters = 32, kernel_size=(3,3),activation = 'sigmoid'),
     layers.BatchNormalization(),
     layers.MaxPooling2D(pool_size=(2, 2)),
     layers.Conv2D(64, kernel_size=(3, 3), activation="sigmoid"),
     layers.BatchNormalization(),
     layers.MaxPooling2D(pool_size=(2, 2)),
     layers.Flatten(),
     layers.Dense(num_classes, activation="softmax"),
    ]
    )
model1.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 94, 94, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 94, 94, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 47, 47, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 45, 45, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 45, 45, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 30976)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │        61,954 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 81,730 (319.26 KB)

 Trainable params: 81,538 (318.51 KB)

 Non-trainable params: 192 (768.00 B)

In [12]:
# training model 2
batch_size = 200
epochs = 20

model1.compile(loss="categorical_crossentropy", optimizer="sgd", metrics=["accuracy", "auc"])

history1 = model1.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

Epoch 1/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 14s 91ms/step - accuracy: 0.5509 - auc: 0.5601 - loss: 13.5801 - val_accuracy: 0.3975 - val_auc: 0.3975 - val_loss: 34.3163
Epoch 2/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step - accuracy: 0.6375 - auc: 0.6663 - loss: 7.0650 - val_accuracy: 0.3975 - val_auc: 0.4336 - val_loss: 3.7938
Epoch 3/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step - accuracy: 0.6928 - auc: 0.7433 - loss: 1.9080 - val_accuracy: 0.4861 - val_auc: 0.5694 - val_loss: 1.6228
Epoch 4/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 55ms/step - accuracy: 0.7262 - auc: 0.7894 - loss: 1.3929 - val_accuracy: 0.6533 - val_auc: 0.7502 - val_loss: 0.8198
Epoch 5/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 55ms/step - accuracy: 0.7382 - auc: 0.8099 - loss: 1.0675 - val_accuracy: 0.7347 - val_auc: 0.8061 - val_loss: 0.9822
Epoch 6/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 54ms/step - accuracy: 0.7498 - auc: 0.8190 - loss: 0.9794 - val_accuracy: 0.6174 - val_auc: 0.6559 - val_loss: 3.6386
Epoch 7/20
100/100 ━━━━━━━━━━━━

In [13]:
# Model 2: Writing away the results 
write_away('Basic_CNN_St_no', history1.history, 'Basic_CNN_St_no.csv')

,accuracy,auc,loss,val_accuracy,val_auc,val_loss
0,0.596233,0.610591,9.318644,0.397547,0.397547,34.316303
1,0.665724,0.707847,3.605817,0.397547,0.433603,3.793843
2,0.694460,0.752296,1.702577,0.486143,0.569393,1.622821
3,0.723095,0.788167,1.234016,0.653339,0.750158,0.819766
4,0.745063,0.815980,0.967526,0.734666,0.806053,0.982167
5,0.767436,0.839294,0.786480,0.617447,0.655865,3.638607
6,0.780264,0.849524,0.742823,0.679237,0.775867,1.053467
7,0.801576,0.875043,0.578627,0.702862,0.753169,2.166219
8,0.812585,0.889114,0.516536,0.604725,0.603024,16.299782
9,0.767032,0.830960,0.937898,0.692867,0.742024,2.327729


In [14]:
# Select 20% of observations
train_df = sub_sample(0.2)
train_df

,id,label
101866,8312c9bac5ab0dded0b79b0e8793ac4470727f40,0
104382,146db8073ee4fc6e64dac8cc8b835306ce4f00a5,1
123473,d6bfa926359cdffe8a770c4c6513322924825928,1
201216,9e2bb84236b7adcd4d245dd6ac9d573bea10204b,0
62800,1809061f44efe7f494c72da733ba50f6a5f054c9,0
...,...,...
110944,18b62ca10f10a13b9dcab6c377a69e3afbb4f716,0
118348,74e880c6deb43c4d0a31adba76becb1eebfaa813,1
28330,f2552d8e74f0c4ccead7af168b9c3d5e2ce94bce,0
5173,112e7f5aff9bb19cfe15e8646ba6a5bcf7dbe4fd,0


In [15]:
# Apply function to_array()
train_set = to_array()
x_train = train_set[0]
y_train = train_set[1]

### Model 3 

In [16]:
num_classes = 2
input_shape = (96, 96, 3)

model2 = keras.Sequential(
    [keras.Input(shape = input_shape),
     layers.Conv2D(filters = 32, kernel_size=(3,3),activation = 'sigmoid'),
     layers.BatchNormalization(),
     layers.MaxPooling2D(pool_size=(2, 2)),
     layers.Conv2D(64, kernel_size=(3, 3), activation="sigmoid"),
     layers.BatchNormalization(),
     layers.MaxPooling2D(pool_size=(2, 2)),
     layers.Flatten(),
     layers.Dense(num_classes, activation="softmax"),
    ]
    )
model2.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 94, 94, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 94, 94, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 47, 47, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 45, 45, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 45, 45, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 30976)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │        61,954 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 81,730 (319.26 KB)

 Trainable params: 81,538 (318.51 KB)

 Non-trainable params: 192 (768.00 B)

In [17]:
# training model 3
batch_size = 200
epochs = 20

model2.compile(loss="categorical_crossentropy", optimizer="sgd", metrics=["accuracy", "auc"])

history2 = model2.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

Epoch 1/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 12s 91ms/step - accuracy: 0.5563 - auc: 0.5636 - loss: 12.7753 - val_accuracy: 0.3975 - val_auc: 0.3975 - val_loss: 6.8132
Epoch 2/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 56ms/step - accuracy: 0.6639 - auc: 0.7016 - loss: 2.8747 - val_accuracy: 0.3975 - val_auc: 0.5035 - val_loss: 1.9683
Epoch 3/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 56ms/step - accuracy: 0.7063 - auc: 0.7634 - loss: 1.5685 - val_accuracy: 0.3989 - val_auc: 0.4425 - val_loss: 4.0689
Epoch 4/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 54ms/step - accuracy: 0.7246 - auc: 0.7925 - loss: 1.1876 - val_accuracy: 0.5084 - val_auc: 0.5315 - val_loss: 2.5087
Epoch 5/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step - accuracy: 0.7216 - auc: 0.7841 - loss: 1.1766 - val_accuracy: 0.5348 - val_auc: 0.5674 - val_loss: 2.4147
Epoch 6/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - accuracy: 0.7856 - auc: 0.8600 - loss: 0.6604 - val_accuracy: 0.6979 - val_auc: 0.7799 - val_loss: 1.1175
Epoch 7/20
100/100 ━━━━━━━━━━━━━

In [18]:
# Model 3: Writing away the results 
write_away('Basic_CNN_St_no_20per', history2.history, 'Basic_CNN_St_no_20per.csv')

,accuracy,auc,loss,val_accuracy,val_auc,val_loss
0,0.591536,0.603889,8.773063,0.397547,0.397547,6.813209
1,0.670572,0.713762,2.515066,0.397547,0.503527,1.968340
2,0.711025,0.768689,1.522489,0.398910,0.442545,4.068893
3,0.730519,0.797543,1.128818,0.508405,0.531475,2.508693
4,0.737842,0.806443,0.991746,0.534757,0.567386,2.414720
5,0.774254,0.848063,0.720516,0.697865,0.779866,1.117532
6,0.797030,0.872575,0.561095,0.419355,0.415353,7.672112
7,0.791930,0.859945,0.703248,0.655157,0.700311,2.676844
8,0.780920,0.851359,0.715702,0.457065,0.442492,8.444434
9,0.744356,0.808877,1.210368,0.707860,0.778772,0.887515
